# Fine-tune Hugging Face Wav2Vec2 Gender Recognition on ViVoice/Vi-26 Gender Dataset

Notebook này fine-tune model `alefiury/wav2vec2-large-xlsr-53-gender-recognition-librispeech` trên tập train, sau đó evaluate trên tập test theo cùng logic notebook hiện tại:

1. Đọc metadata từ Excel.
2. Tự dò cột audio và cột giới tính.
3. Match audio trong thư mục dataset.
4. Chuẩn hóa nhãn gender thành `female=0`, `male=1`.
5. Split 80/20 với `random_state=42`.
6. Fine-tune model trên `train_df`.
7. Test trên `test_df` và lưu metrics/predictions/checkpoint.

> Nếu metadata dùng tên cột/giá trị giới tính khác, sửa `path_candidates`, `gender_candidates`, hoặc hàm `normalize_gender()` ở các cell đầu.

In [1]:
# Optional install for Kaggle if needed
# Chạy cell này nếu môi trường thiếu transformers/torchaudio/soundfile/accelerate
# !pip -q install -U transformers torchaudio soundfile accelerate safetensors

In [2]:
# =========================
# 1. Imports & Configuration
# =========================
import os
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torchaudio
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, get_linear_schedule_with_warmup

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Giữ đúng path theo notebook hiện tại.
DATA_DIR = Path('/kaggle/input/datasets/tranvannha/vi-26-dataset/Vi_26/data')
EXCEL_PATH = Path('/kaggle/input/datasets/tranvannha/vi-26-dataset/Vi_26/VBee.xlsx')

MODEL_ID = 'alefiury/wav2vec2-large-xlsr-53-gender-recognition-librispeech'

SR = 16000
MAX_DURATION = 5.0
BATCH_SIZE = 4              # wav2vec2-large khá nặng, tăng lên 8/16 nếu GPU đủ RAM
GRAD_ACCUM_STEPS = 4        # effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS
EPOCHS = 20
LR = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 2
FREEZE_FEATURE_ENCODER = True
USE_AMP = torch.cuda.is_available()

OUTPUT_DIR = Path('/kaggle/working/hf_gender_finetune_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Output dir:', OUTPUT_DIR)

Device: cuda
Output dir: /kaggle/working/hf_gender_finetune_results


In [3]:
# =========================
# 2. Load metadata and detect columns
# =========================
df = pd.read_excel(EXCEL_PATH)
df.columns = [str(c).strip() for c in df.columns]

path_candidates = ['Path', 'path', 'Audio', 'audio', 'audio_path', 'File', 'file', 'Filename', 'filename']
gender_candidates = [
    'Gender', 'gender', 'Sex', 'sex', 'Speaker Gender', 'speaker_gender',
    'Giới tính', 'Gioi tinh', 'gioi_tinh', 'Gender ', 'Nam/Nữ'
]

path_col = next((c for c in path_candidates if c in df.columns), None)
gender_col = next((c for c in gender_candidates if c in df.columns), None)

if path_col is None:
    raise ValueError(f'Không tìm thấy cột đường dẫn audio. Columns hiện có: {df.columns.tolist()}')

if gender_col is None:
    raise ValueError(f'Không tìm thấy cột giới tính. Columns hiện có: {df.columns.tolist()}')

print('Path column  :', path_col)
print('Gender column:', gender_col)

df = df.dropna(subset=[path_col, gender_col]).copy()
df['audio_basename'] = df[path_col].apply(lambda x: Path(str(x).replace('\\', '/')).name)

print('Metadata shape after dropna:', df.shape)
print('Raw gender distribution:')
display(df[gender_col].astype(str).str.strip().value_counts())
display(df.head())

Path column  : Path
Gender column: Gender
Metadata shape after dropna: (547, 16)
Raw gender distribution:


Gender
Male      382
Female    165
Name: count, dtype: int64

,Id,Speaker,Path,Transcript,Province,Region,Gender,Age-group,Duration (s),Local word,Loanword,Total word,Field,Unnamed: 13,Unnamed: 14,audio_basename
0,north_spk_1,Bảo Trung Review phim,D:\Viettel\dataset\my_contribution\baotrung_rv...,Hôm nay tôi cập nhật nhanh về gió mùa Đông Bắc...,NinhBinh,North,Male,Adolescent,40,3 (khu bán đọi; cơm-cháy; thế-lào),3 (in-tơ-nét; teamwork; phây-búc),148.0,Weather,Review North,NaN,baotrung_rv_phim.mp3
1,north_spk_2,Thuyết Minh Phim,D:\Viettel\dataset\my_contribution\tmfilm_fema...,"Trong cuộc họp hôm nay, tôi cập nhật phân công...",ThaiNguyen,North,Female,Adolescent,40,3 (đồi-chè; chợ-phiên; chè-móc-câu),0,144.0,meeting,NaN,NaN,tmfilm_female.mp3
2,north_spk_3,Kiên Xoăn,D:\Viettel\dataset\my_contribution\kienxoan_hn...,"Tôi gọi video để nói về thăm hỏi, tiện nhắc lu...",HaNoi,North,Male,Adolescent,40,2 (cốm-Vòng;bún-thang),0,146.0,video_call,NaN,Prompt,kienxoan_hn.mp3
3,north_spk_4,Ngọc Vy,D:\Viettel\dataset\my_contribution\ngocvy_ninh...,Tôi kể chuyện tào lao một chút về đồ ăn vặt rồ...,NinhBinh,North,Female,Adolescent,41,4 (mua nà; Tràng-An; cơm-cháy;thế-lào),2 (rating; tóp-tóp),153.0,casual_conversation,NaN,\nVới yêu cầu tạo transcript của MỘT NGƯỜI NÓI...,ngocvy_ninhbinh.mp3
4,north_spk_5,Vuive,D:\Viettel\dataset\my_contribution\vuive.mp3,Xin thông báo: liên quan đến điều chỉnh lịch v...,PhuTho,North,Male,Adolescent,39,3 (bánh-tai; đền-Hùng; hát-xoan),2 (ship; deal),146.0,announcement,NaN,NaN,vuive.mp3


In [4]:
# =========================
# 3. Match metadata rows with actual audio files
# =========================
if not DATA_DIR.exists():
    raise FileNotFoundError(f'DATA_DIR does not exist: {DATA_DIR}')

audio_exts = {'.wav', '.mp3', '.flac', '.m4a', '.ogg', '.aac'}
audio_files = [p for p in DATA_DIR.rglob('*') if p.suffix.lower() in audio_exts]
file_index = {p.name: p for p in audio_files}

print('Audio files found:', len(audio_files))

df['audio_path'] = df['audio_basename'].map(lambda name: file_index.get(name))

# Fallback: match by stem if extension/name differs slightly
stem_index = {}
for p in audio_files:
    stem_index.setdefault(p.stem, p)

df.loc[df['audio_path'].isna(), 'audio_path'] = df.loc[df['audio_path'].isna(), 'audio_basename'].map(
    lambda name: stem_index.get(Path(str(name)).stem)
)

missing = df['audio_path'].isna().sum()
print('Matched rows:', len(df) - missing)
print('Missing audio rows:', missing)

if missing > 0:
    display(df.loc[df['audio_path'].isna(), [path_col, 'audio_basename', gender_col]].head(20))

df = df.dropna(subset=['audio_path']).copy()
df['audio_path'] = df['audio_path'].astype(str)

if len(df) == 0:
    raise ValueError('No audio files matched. Please check DATA_DIR and filename mapping.')

Audio files found: 547
Matched rows: 545
Missing audio rows: 2


,Path,audio_basename,Gender
55,D:\Viettel\dataset\my_contribution\PhuTho01.mp3,PhuTho01.mp3,Male
341,D:\Viettel\dataset\my_contribution\nguyeQuangT...,nguyeQuangTri.mp3,Male


In [5]:
# =========================
# 4. Normalize gender labels and split train/test 80/20
# =========================
def normalize_gender(x):
    s = str(x).strip().lower()
    # Normalize common Vietnamese characters enough for gender labels
    trans = str.maketrans({
        'á':'a','à':'a','ả':'a','ã':'a','ạ':'a','ă':'a','ắ':'a','ằ':'a','ẳ':'a','ẵ':'a','ặ':'a','â':'a','ấ':'a','ầ':'a','ẩ':'a','ẫ':'a','ậ':'a',
        'é':'e','è':'e','ẻ':'e','ẽ':'e','ẹ':'e','ê':'e','ế':'e','ề':'e','ể':'e','ễ':'e','ệ':'e',
        'í':'i','ì':'i','ỉ':'i','ĩ':'i','ị':'i',
        'ó':'o','ò':'o','ỏ':'o','õ':'o','ọ':'o','ô':'o','ố':'o','ồ':'o','ổ':'o','ỗ':'o','ộ':'o','ơ':'o','ớ':'o','ờ':'o','ở':'o','ỡ':'o','ợ':'o',
        'ú':'u','ù':'u','ủ':'u','ũ':'u','ụ':'u','ư':'u','ứ':'u','ừ':'u','ử':'u','ữ':'u','ự':'u',
        'ý':'y','ỳ':'y','ỷ':'y','ỹ':'y','ỵ':'y',
        'đ':'d',
    })
    s = s.translate(trans)

    male_values = {'male', 'm', 'man', 'men', 'nam', 'boy', '1'}
    female_values = {'female', 'f', 'woman', 'women', 'nu', 'girl', '0'}

    if s in male_values:
        return 'male'
    if s in female_values:
        return 'female'
    return None

LABEL2ID = {'female': 0, 'male': 1}
ID2LABEL = {0: 'female', 1: 'male'}

df['gender_norm'] = df[gender_col].apply(normalize_gender)
unknown = df['gender_norm'].isna().sum()
print('Rows with unknown gender labels:', unknown)
if unknown > 0:
    display(df.loc[df['gender_norm'].isna(), [gender_col, path_col]].head(20))

df = df.dropna(subset=['gender_norm']).copy()
df['label'] = df['gender_norm'].map(LABEL2ID).astype(int)

print('Normalized gender distribution:')
display(df['gender_norm'].value_counts())
print('Encoded distribution:', Counter(df['label']))

counts = df['label'].value_counts()
stratify_labels = df['label'] if counts.min() >= 2 else None
if stratify_labels is None:
    print('WARNING: At least one class has <2 samples, using non-stratified split.')

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=stratify_labels,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Train size: {len(train_df)} ({len(train_df)/len(df):.1%})')
print(f'Test size : {len(test_df)} ({len(test_df)/len(df):.1%})')
print('Train distribution:')
display(train_df['gender_norm'].value_counts())
print('Test distribution:')
display(test_df['gender_norm'].value_counts())

Rows with unknown gender labels: 0
Normalized gender distribution:


gender_norm
male      380
female    165
Name: count, dtype: int64

Encoded distribution: Counter({1: 380, 0: 165})
Train size: 436 (80.0%)
Test size : 109 (20.0%)
Train distribution:


gender_norm
male      304
female    132
Name: count, dtype: int64

Test distribution:


gender_norm
male      76
female    33
Name: count, dtype: int64

In [6]:
# =========================
# 5. Dataset and DataLoaders
# =========================
def load_audio_tensor(path, sr=SR, max_duration=MAX_DURATION):
    wav, orig_sr = torchaudio.load(path)

    # mono
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    # resample
    if orig_sr != sr:
        wav = torchaudio.transforms.Resample(orig_sr, sr)(wav)

    # pad/truncate to MAX_DURATION seconds
    target_len = int(sr * max_duration)
    cur_len = wav.shape[1]
    if cur_len < target_len:
        wav = F.pad(wav, (0, target_len - cur_len))
    elif cur_len > target_len:
        wav = wav[:, :target_len]

    return wav.squeeze(0).numpy()


class GenderAudioDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio = load_audio_tensor(row['audio_path'])
        label = int(row['label'])
        return {
            'audio': audio,
            'label': label,
            'audio_path': row['audio_path'],
        }


feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)


def collate_fn(batch):
    audios = [item['audio'] for item in batch]
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)
    paths = [item['audio_path'] for item in batch]

    inputs = feature_extractor(
        audios,
        sampling_rate=SR,
        return_tensors='pt',
        padding=True,
        return_attention_mask=True,
    )
    inputs['labels'] = labels
    inputs['audio_paths'] = paths
    return inputs


train_dataset = GenderAudioDataset(train_df)
test_dataset = GenderAudioDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
    drop_last=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
)

sample = train_dataset[0]
print('Sample audio shape:', sample['audio'].shape, '| Label:', sample['label'], ID2LABEL[sample['label']])
print('Train batches:', len(train_loader), '| Test batches:', len(test_loader))

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

Sample audio shape: (80000,) | Label: 1 male
Train batches: 109 | Test batches: 28


In [7]:
# =========================
# 6. Load pretrained model for fine-tuning
# =========================
model = AutoModelForAudioClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=False,
)

if FREEZE_FEATURE_ENCODER and hasattr(model, 'freeze_feature_encoder'):
    model.freeze_feature_encoder()
    print('Feature encoder frozen.')

model.to(DEVICE)
print('Loaded:', MODEL_ID)
print('Model config id2label:', model.config.id2label)

# Optional: reduce memory when supported
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()
    print('Gradient checkpointing enabled.')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Feature encoder frozen.
Loaded: alefiury/wav2vec2-large-xlsr-53-gender-recognition-librispeech
Model config id2label: {0: 'female', 1: 'male'}
Gradient checkpointing enabled.


In [8]:
# =========================
# 7. Optimizer, scheduler, train/evaluate functions
# =========================
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

num_update_steps_per_epoch = int(np.ceil(len(train_loader) / GRAD_ACCUM_STEPS))
num_training_steps = EPOCHS * num_update_steps_per_epoch
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


def move_batch_to_device(batch):
    labels = batch.pop('labels').to(DEVICE, non_blocking=True)
    paths = batch.pop('audio_paths')
    batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
    return batch, labels, paths


def train_one_epoch(model, loader, epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0
    total_samples = 0

    pbar = tqdm(loader, desc=f'Epoch {epoch}/{EPOCHS} - train')
    for step, batch in enumerate(pbar, start=1):
        batch, labels, _ = move_batch_to_device(batch)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(**batch, labels=labels)
            loss = outputs.loss
            loss_for_backward = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss_for_backward).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_samples += bs
        pbar.set_postfix(loss=total_loss / max(total_samples, 1), lr=scheduler.get_last_lr()[0])

    return total_loss / max(total_samples, 1)


@torch.no_grad()
def evaluate(model, loader, desc='evaluate'):
    model.eval()
    all_probs, all_preds, all_targets, all_paths = [], [], [], []
    total_loss = 0.0
    total_samples = 0

    for batch in tqdm(loader, desc=desc):
        batch, labels, paths = move_batch_to_device(batch)

        outputs = model(**batch, labels=labels)
        logits = outputs.logits
        loss = outputs.loss
        probs = torch.softmax(logits, dim=-1)
        preds = probs.argmax(dim=-1)

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_samples += bs
        all_probs.append(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(labels.cpu().numpy().tolist())
        all_paths.extend(paths)

    return {
        'loss': total_loss / max(total_samples, 1),
        'probs': np.vstack(all_probs),
        'preds': np.array(all_preds),
        'targets': np.array(all_targets),
        'paths': all_paths,
    }


def compute_metrics(pred_out, prefix='Test'):
    y_true = pred_out['targets']
    y_pred = pred_out['preds']
    return {
        f'{prefix} Loss': pred_out['loss'],
        f'{prefix} Accuracy': accuracy_score(y_true, y_pred),
        f'{prefix} Precision_macro': precision_score(y_true, y_pred, labels=[0, 1], average='macro', zero_division=0),
        f'{prefix} Recall_macro': recall_score(y_true, y_pred, labels=[0, 1], average='macro', zero_division=0),
        f'{prefix} F1_macro': f1_score(y_true, y_pred, labels=[0, 1], average='macro', zero_division=0),
        f'{prefix} Precision_weighted': precision_score(y_true, y_pred, labels=[0, 1], average='weighted', zero_division=0),
        f'{prefix} Recall_weighted': recall_score(y_true, y_pred, labels=[0, 1], average='weighted', zero_division=0),
        f'{prefix} F1_weighted': f1_score(y_true, y_pred, labels=[0, 1], average='weighted', zero_division=0),
    }

print('Training steps:', num_training_steps, '| Warmup steps:', num_warmup_steps)

Training steps: 560 | Warmup steps: 56


In [9]:
# =========================
# 8. Fine-tune on train set
# =========================
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, epoch)
    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'lr': scheduler.get_last_lr()[0],
    })
    print(f'Epoch {epoch}/{EPOCHS} - Train Loss: {train_loss:.4f}')

history_df = pd.DataFrame(history)
display(history_df)

history_path = OUTPUT_DIR / 'finetune_history.csv'
history_df.to_csv(history_path, index=False)
print('Saved training history:', history_path)

Epoch 1/20 - train:   0%|          | 0/109 [00:00<?, ?it/s]

Epoch 1/20 - Train Loss: 0.6111


Epoch 2/20 - train:   0%|          | 0/109 [00:00<?, ?it/s]

Epoch 2/20 - Train Loss: 0.5105


Epoch 3/20 - train:   0%|          | 0/109 [00:00<?, ?it/s]

...

Epoch 19/20 - Train Loss: 0.2555


Epoch 20/20 - train:   0%|          | 0/109 [00:00<?, ?it/s]

Epoch 20/20 - Train Loss: 0.2587


,epoch,train_loss,lr
0,1,0.611094,5.000000e-06
1,2,0.510512,1.000000e-05
2,3,0.401097,9.444444e-06
3,4,0.291933,8.888889e-06
4,5,0.286698,8.333333e-06
5,6,0.280080,7.777778e-06
6,7,0.271037,7.222222e-06
7,8,0.265073,6.666667e-06
8,9,0.267147,6.111111e-06
9,10,0.253347,5.555556e-06


Saved training history: /kaggle/working/hf_gender_finetune_results/finetune_history.csv


In [10]:
# =========================
# 9. Evaluate final fine-tuned model on test set
# =========================
pred_out = evaluate(model, test_loader, desc='final test')
y_true = pred_out['targets']
y_pred = pred_out['preds']

metrics = {
    'Model': MODEL_ID,
    'Fine-tuned': True,
    'Epochs': EPOCHS,
    'Train size': len(train_df),
    'Test size': len(test_df),
}
metrics.update(compute_metrics(pred_out, prefix='Test'))

metrics_df = pd.DataFrame([metrics])
print('Final test metrics:')
display(metrics_df)

print('Classification report:')
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=['female', 'male'],
    zero_division=0,
))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(cm, index=['true_female', 'true_male'], columns=['pred_female', 'pred_male'])
print('Confusion matrix:')
display(cm_df)

final test:   0%|          | 0/28 [00:00<?, ?it/s]

Final test metrics:


,Model,Fine-tuned,Epochs,Train size,Test size,Test Loss,Test Accuracy,Test Precision_macro,Test Recall_macro,Test F1_macro,Test Precision_weighted,Test Recall_weighted,Test F1_weighted
0,alefiury/wav2vec2-large-xlsr-53-gender-recogni...,True,20,436,109,0.262523,0.926606,0.913078,0.913078,0.913078,0.926606,0.926606,0.926606


Classification report:
              precision    recall  f1-score   support

      female       0.88      0.88      0.88        33
        male       0.95      0.95      0.95        76

    accuracy                           0.93       109
   macro avg       0.91      0.91      0.91       109
weighted avg       0.93      0.93      0.93       109

Confusion matrix:


,pred_female,pred_male
true_female,29,4
true_male,4,72


In [11]:
# =========================
# 10. Save predictions, metrics, and fine-tuned checkpoint
# =========================
probs = pred_out['probs']

pred_df = test_df.copy()
pred_df['true_label_id'] = pred_out['targets']
pred_df['true_gender'] = [ID2LABEL[int(x)] for x in pred_out['targets']]
pred_df['pred_label_id'] = pred_out['preds']
pred_df['pred_gender'] = [ID2LABEL[int(x)] for x in pred_out['preds']]
pred_df['prob_female'] = probs[:, 0]
pred_df['prob_male'] = probs[:, 1]
pred_df['correct'] = pred_df['true_label_id'] == pred_df['pred_label_id']

metrics_path = OUTPUT_DIR / 'finetuned_gender_metrics.csv'
preds_path = OUTPUT_DIR / 'finetuned_gender_predictions.csv'
cm_path = OUTPUT_DIR / 'finetuned_gender_confusion_matrix.csv'
checkpoint_dir = OUTPUT_DIR / 'finetuned_checkpoint'

metrics_df.to_csv(metrics_path, index=False)
pred_df.to_csv(preds_path, index=False)
cm_df.to_csv(cm_path, index=True)

model.save_pretrained(checkpoint_dir)
feature_extractor.save_pretrained(checkpoint_dir)

print('Saved outputs to:', OUTPUT_DIR)
print('Metrics:', metrics_path)
print('Predictions:', preds_path)
print('Confusion matrix:', cm_path)
print('Fine-tuned checkpoint:', checkpoint_dir)

display(pred_df[[path_col, gender_col, 'audio_path', 'true_gender', 'pred_gender', 'prob_female', 'prob_male', 'correct']].head(20))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved outputs to: /kaggle/working/hf_gender_finetune_results
Metrics: /kaggle/working/hf_gender_finetune_results/finetuned_gender_metrics.csv
Predictions: /kaggle/working/hf_gender_finetune_results/finetuned_gender_predictions.csv
Confusion matrix: /kaggle/working/hf_gender_finetune_results/finetuned_gender_confusion_matrix.csv
Fine-tuned checkpoint: /kaggle/working/hf_gender_finetune_results/finetuned_checkpoint


,Path,Gender,audio_path,true_gender,pred_gender,prob_female,prob_male,correct
0,D:\Viettel\dataset\my_contribution\south_revie...,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.039695,0.960305,True
1,D:\Viettel\dataset\my_contribution\north_news_...,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.036628,0.963372,True
2,D:\Viettel\dataset\my_contribution\south_story...,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.031853,0.968147,True
3,D:\Viettel\dataset\my_contribution\north_news_...,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.046209,0.953791,True
4,D:\Viettel\dataset\my_contribution\drthanh.mp3,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.037916,0.962084,True
5,D:\Viettel\dataset\my_contribution\north_story...,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.031491,0.968509,True
6,D:\Viettel\dataset\my_contribution\north_news_...,Female,/kaggle/input/datasets/tranvannha/vi-26-datase...,female,female,0.605819,0.394181,True
7,D:\Viettel\dataset\my_contribution\ngocvy_ninh...,Female,/kaggle/input/datasets/tranvannha/vi-26-datase...,female,female,0.892725,0.107275,True
8,D:\Viettel\dataset\my_contribution\north_story...,Male,/kaggle/input/datasets/tranvannha/vi-26-datase...,male,male,0.032257,0.967743,True
9,D:\Viettel\dataset\my_contribution\south_story...,Female,/kaggle/input/datasets/tranvannha/vi-26-datase...,female,male,0.398531,0.601469,False


## Gợi ý khi chạy trên Kaggle

- Nếu bị CUDA OOM, giảm `BATCH_SIZE` xuống `2` hoặc `1`, giữ `GRAD_ACCUM_STEPS` để bù effective batch size.
- Nếu train quá chậm, giảm `MAX_DURATION` từ `5.0` xuống `3.0`, hoặc giảm `EPOCHS`.
- Nếu muốn fine-tune toàn bộ encoder mạnh hơn, đặt `FREEZE_FEATURE_ENCODER = False`, nhưng sẽ tốn GPU RAM hơn.
- Kết quả cuối cùng nằm trong `/kaggle/working/hf_gender_finetune_results`.